In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.silver.orders")

dq_summary = orders.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("missing_order_id"),
    F.sum(
        F.when(F.col("customer_id").isNull(), 1).otherwise(0)
    ).alias("missing_customer_id"),
    F.sum(
        F.when(F.col("total_amount") < 0, 1).otherwise(0)
    ).alias("negative_amounts")
)

dq_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.data_quality_summary")

display(dq_summary)